# 📈 Aula 14 — Regressão e Previsão de Valores

## Utilizando dados para prever valores numéricos

**Disciplina:** ISW-039 — Mineração de Dados  
**Curso:** Desenvolvimento de Software Multiplataforma (DSM)  
**Ambiente:** Google Colab  
**Linguagem:** Python  
**Bibliotecas:** Pandas, NumPy, Matplotlib e Scikit-learn

---

## 🎯 Objetivos da aula

Ao final desta aula, você deverá ser capaz de:

- Compreender a diferença entre classificação e regressão;
- Identificar problemas que envolvem previsão de valores numéricos;
- Definir variáveis preditoras e variável alvo;
- Separar dados para treinamento e teste;
- Aplicar Regressão Linear;
- Visualizar uma relação entre variáveis;
- Fazer previsões numéricas;
- Avaliar um modelo de regressão;
- Compreender MAE, MSE, RMSE e R²;
- Interpretar erros de previsão;
- Identificar limitações de um modelo de regressão;
- Aplicar regressão ao projeto individual.

> **Projeto didático:** continuaremos utilizando o monitoramento de motores elétricos. Desta vez, em vez de prever **Normal/Falha**, queremos prever um **valor numérico**, como a temperatura do motor.


# 🧠 1. Classificação × Regressão

Nas aulas anteriores trabalhamos com classificação.

Exemplo:

```text
Temperatura
Vibração
Corrente
      ↓
  CLASSIFICADOR
      ↓
Normal / Falha
```

Agora queremos prever um número.

Exemplo:

```text
Vibração
Corrente
RPM
Tensão
      ↓
   REGRESSÃO
      ↓
Temperatura prevista
```

### Classificação

A saída é uma **categoria**.

```text
Normal
Falha
A
B
C
```

### Regressão

A saída é um **valor numérico**.

```text
65.2 °C
72.8 °C
81.4 °C
```

Essa diferença é fundamental.


# 🏭 2. Problema industrial

Imagine que o sistema de monitoramento possui os seguintes sensores:

- vibração;
- corrente;
- tensão;
- RPM;
- temperatura.

Queremos responder:

> **Qual será a temperatura estimada de um motor a partir das demais variáveis?**

Isso pode ser utilizado, por exemplo, para:

- estimar comportamento térmico;
- antecipar condições anormais;
- apoiar manutenção;
- analisar influência de variáveis;
- gerar indicadores para sistemas de monitoramento.


# 📐 3. O que é Regressão Linear?

A regressão linear tenta encontrar uma relação matemática entre variáveis.

No caso mais simples:

```text
y = a + bx
```

Onde:

```text
y → valor que queremos prever
x → variável utilizada para previsão
a → intercepto
b → coeficiente
```

Visualmente:

```text
Temperatura
    ↑
    |              ●
    |          ●
    |       ●
    |    ●
    | ●
    +----------------------→ Vibração
```

O modelo procura uma linha que represente da melhor maneira possível a relação entre os dados.


# 💻 4. Preparando o ambiente

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

np.random.seed(42)

print("Ambiente preparado!")

# 📥 5. Criando a base de motores

Vamos criar dados simulados.

A temperatura será influenciada por:

- vibração;
- corrente;
- RPM;
- tensão.

Isso permitirá criar um problema de regressão.


In [ ]:
n = 600

df = pd.DataFrame({
    "vibracao": np.random.normal(2.2, 0.7, n),
    "corrente": np.random.normal(13, 2, n),
    "tensao": np.random.normal(380, 4, n),
    "rpm": np.random.normal(1740, 15, n)
})

df["temperatura"] = (
    35
    + 4.5 * df["vibracao"]
    + 1.8 * df["corrente"]
    - 0.015 * df["rpm"]
    + 0.05 * (df["tensao"] - 380)
    + np.random.normal(0, 2.5, n)
)

df.head()

Vamos analisar as estatísticas.


In [ ]:
df.describe().round(2)

# 📊 6. Explorando a relação entre variáveis

Vamos começar observando:

```text
vibração × temperatura
```


In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    df["vibracao"],
    df["temperatura"],
    alpha=0.6
)

plt.title("Vibração × Temperatura")
plt.xlabel("Vibração")
plt.ylabel("Temperatura (°C)")
plt.show()

Visualmente podemos investigar se existe uma tendência.

Agora vamos observar corrente × temperatura.


In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    df["corrente"],
    df["temperatura"],
    alpha=0.6
)

plt.title("Corrente × Temperatura")
plt.xlabel("Corrente")
plt.ylabel("Temperatura (°C)")
plt.show()

# 🔗 7. Correlação

Podemos utilizar a correlação para explorar relações lineares entre variáveis.



In [ ]:
df.corr(numeric_only=True).round(2)

A correlação é útil para exploração, mas atenção:

> **Correlação não significa causalidade.**

Uma variável pode estar associada a outra sem necessariamente ser a causa.


# 🎯 8. Definindo X e y

Nosso objetivo é prever:

```text
temperatura
```

Portanto:

```text
X = vibração
    corrente
    tensão
    rpm

y = temperatura
```


In [ ]:
X = df[[
    "vibracao",
    "corrente",
    "tensao",
    "rpm"
]]

y = df["temperatura"]

print("Features:", X.columns.tolist())
print("Target:", y.name)

# ✂️ 9. Separando treino e teste

Vamos utilizar:

```text
80% → treinamento
20% → teste
```


In [ ]:
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Treino:", X_treino.shape)
print("Teste:", X_teste.shape)

# 🤖 10. Treinando a Regressão Linear



In [ ]:
modelo = LinearRegression()

modelo.fit(
    X_treino,
    y_treino
)

print("Modelo treinado!")

# 📐 11. Conhecendo os coeficientes

A regressão encontrou uma equação aproximada.

Vamos observar os coeficientes.


In [ ]:
coeficientes = pd.DataFrame({
    "variavel": X.columns,
    "coeficiente": modelo.coef_
})

coeficientes

Também podemos observar o intercepto:


In [ ]:
print("Intercepto:", round(modelo.intercept_, 3))

A interpretação simplificada é:

```text
Temperatura =
intercepto
+
coeficiente_vibracao × vibração
+
coeficiente_corrente × corrente
+
...
```

Os coeficientes indicam como o modelo utiliza as variáveis para realizar as previsões.

Mas cuidado:

> Um coeficiente não deve ser interpretado automaticamente como uma relação causal.


# 🔮 12. Fazendo previsões



In [ ]:
y_pred = modelo.predict(X_teste)

y_pred[:10]

Vamos comparar valores reais e previstos.


In [ ]:
comparacao = pd.DataFrame({
    "real": y_teste.values,
    "previsto": y_pred
})

comparacao["erro"] = (
    comparacao["real"] - comparacao["previsto"]
)

comparacao.head(15).round(2)

# 📊 13. Gráfico — Real × Previsto

Um gráfico bastante útil é comparar:

```text
valor real
     ×
valor previsto
```


In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    y_teste,
    y_pred,
    alpha=0.6
)

limite_min = min(y_teste.min(), y_pred.min())
limite_max = max(y_teste.max(), y_pred.max())

plt.plot(
    [limite_min, limite_max],
    [limite_min, limite_max],
    linestyle="--"
)

plt.title("Temperatura Real × Temperatura Prevista")
plt.xlabel("Temperatura real (°C)")
plt.ylabel("Temperatura prevista (°C)")
plt.show()

A linha diagonal representa:

```text
previsto = real
```

Quanto mais próximos os pontos estiverem dessa linha, melhor tende a ser a previsão.


# 📏 14. MAE — Erro Absoluto Médio

O **MAE (Mean Absolute Error)** calcula a média dos erros absolutos.

Exemplo:

```text
Erro 1 = 2 °C
Erro 2 = 1 °C
Erro 3 = 3 °C

MAE = (2 + 1 + 3) / 3
    = 2 °C
```

Isso torna a interpretação bastante intuitiva:

> Em média, o modelo errou aproximadamente X graus.


In [ ]:
mae = mean_absolute_error(y_teste, y_pred)

print(f"MAE: {mae:.2f} °C")

# 📐 15. MSE — Erro Quadrático Médio

O MSE eleva os erros ao quadrado.

Isso faz com que erros grandes tenham peso maior.

```text
MSE = média dos erros²
```

Vamos calcular.


In [ ]:
mse = mean_squared_error(y_teste, y_pred)

print(f"MSE: {mse:.2f}")

# 📏 16. RMSE — Raiz do Erro Quadrático Médio

O RMSE é a raiz quadrada do MSE.

Sua vantagem é voltar para a unidade original da variável.

Se estamos prevendo temperatura:

```text
RMSE → °C
```


In [ ]:
rmse = np.sqrt(mse)

print(f"RMSE: {rmse:.2f} °C")

# 📈 17. R² — Coeficiente de determinação

O R² indica quanto da variação da variável alvo é explicada pelo modelo, dentro do conjunto avaliado.

De forma simplificada:

```text
R² próximo de 1
→ modelo explica grande parte da variação

R² próximo de 0
→ modelo explica pouco da variação
```

Também podemos encontrar valores negativos em determinados cenários de teste.


In [ ]:
r2 = r2_score(y_teste, y_pred)

print(f"R²: {r2:.3f}")

# 📊 18. Avaliação completa

Vamos reunir as principais métricas.


In [ ]:
metricas = pd.DataFrame({
    "Métrica": ["MAE", "MSE", "RMSE", "R²"],
    "Valor": [mae, mse, rmse, r2]
})

metricas.round(3)

### Como interpretar?

**MAE**

> Erro médio em unidades da variável.

**MSE**

> Penaliza mais fortemente erros grandes.

**RMSE**

> Erro típico em unidades da variável, com maior influência dos erros grandes.

**R²**

> Indica o quanto da variação dos dados é explicada pelo modelo.


# 🔎 19. Analisando os erros

Vamos criar uma coluna de erro.


In [ ]:
erros = pd.DataFrame({
    "real": y_teste.values,
    "previsto": y_pred
})

erros["erro"] = erros["real"] - erros["previsto"]
erros["erro_absoluto"] = erros["erro"].abs()

erros.head().round(2)

Vamos observar a distribuição dos erros.


In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(erros["erro"], bins=25)

plt.title("Distribuição dos Erros")
plt.xlabel("Erro (°C)")
plt.ylabel("Quantidade")
plt.show()

Um modelo bem ajustado tende a produzir erros concentrados próximos de zero.

Ainda assim, precisamos investigar os maiores erros.


In [ ]:
erros.sort_values(
    "erro_absoluto",
    ascending=False
).head(10).round(2)

Esses registros podem indicar:

- situações incomuns;
- outliers;
- dados de baixa qualidade;
- comportamento não capturado pelo modelo;
- necessidade de novas variáveis.


# 📈 20. Resíduos

Podemos observar os resíduos em relação às previsões.


In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    y_pred,
    erros["erro"],
    alpha=0.6
)

plt.axhline(0, linestyle="--")

plt.title("Resíduos × Valores Preditos")
plt.xlabel("Temperatura prevista")
plt.ylabel("Erro")
plt.show()

A análise dos resíduos ajuda a investigar se existem padrões que o modelo não conseguiu capturar.

Se os erros apresentarem um padrão claro, pode existir alguma estrutura não modelada.


# 🧪 21. Prevendo uma nova leitura

Imagine que o sistema recebeu:

```text
Vibração = 3.0
Corrente = 15.0
Tensão = 380
RPM = 1735
```

Qual temperatura podemos estimar?


In [ ]:
novo_motor = pd.DataFrame({
    "vibracao": [3.0],
    "corrente": [15.0],
    "tensao": [380],
    "rpm": [1735]
})

temperatura_prevista = modelo.predict(novo_motor)

print(
    f"Temperatura prevista: {temperatura_prevista[0]:.2f} °C"
)

Esse é o princípio de um sistema preditivo:

```text
NOVOS DADOS
    ↓
MODELO TREINADO
    ↓
PREVISÃO
    ↓
DECISÃO
```


# 🧠 22. Uma única variável × várias variáveis

Até aqui utilizamos:

```text
vibração
corrente
tensão
rpm
```

Mas poderíamos tentar prever a temperatura utilizando somente uma variável.

Por exemplo:

```text
vibração → temperatura
```

Vamos comparar.


In [ ]:
X_vibracao = df[["vibracao"]]
y = df["temperatura"]

Xv_treino, Xv_teste, yv_treino, yv_teste = train_test_split(
    X_vibracao,
    y,
    test_size=0.20,
    random_state=42
)

modelo_vibracao = LinearRegression()

modelo_vibracao.fit(
    Xv_treino,
    yv_treino
)

pred_vibracao = modelo_vibracao.predict(Xv_teste)

print(
    "R² utilizando apenas vibração:",
    round(r2_score(yv_teste, pred_vibracao), 3)
)

print(
    "MAE utilizando apenas vibração:",
    round(mean_absolute_error(yv_teste, pred_vibracao), 2)
)

Agora compare com o modelo que utiliza várias variáveis.

Pergunta:

> Utilizar mais variáveis sempre garante um modelo melhor?

Não necessariamente.

As variáveis precisam contribuir com informação útil.


# ⚠️ 23. Limitações da Regressão Linear

A regressão linear procura relações lineares.

Mas o mundo real pode ser mais complexo.

Por exemplo:

```text
Temperatura
   ↑
   |       ●
   |    ●
   |  ●
   | ●
   |    ●
   +----------------→ Vibração
```

Se a relação for muito não linear, uma linha pode não representar bem o comportamento.

Outros modelos podem ser necessários.

Também precisamos considerar:

- outliers;
- variáveis irrelevantes;
- relações não lineares;
- dados insuficientes;
- extrapolação;
- qualidade dos sensores.


# 📝 24. Exercícios

## Exercício 1 — Conceitos

Explique a diferença entre:

**classificação** e **regressão**.

Dê um exemplo de cada.


In [ ]:
# Sua resposta



## Exercício 2 — Variáveis

No problema de previsão de temperatura:

- quais são as features?
- qual é o target?


In [ ]:
# Sua resposta



## Exercício 3 — Correlação

Analise a matriz de correlação.

Quais variáveis apresentam maior correlação com temperatura?


In [ ]:
# Sua resposta



## Exercício 4 — Modelo

Treine uma Regressão Linear utilizando apenas:

```text
corrente
```

para prever temperatura.

Calcule:

- MAE;
- RMSE;
- R².


In [ ]:
# Sua resposta



## Exercício 5 — Modelo com duas variáveis

Utilize:

```text
corrente
vibracao
```

Compare com o modelo do exercício anterior.


In [ ]:
# Sua resposta



## Exercício 6 — Todas as variáveis

Treine o modelo utilizando:

```text
vibracao
corrente
tensao
rpm
```

Compare os resultados.


In [ ]:
# Sua resposta



## Exercício 7 — Real × Previsto

Crie o gráfico de valores reais contra valores previstos.

Os pontos estão próximos da diagonal?


In [ ]:
# Sua resposta



## Exercício 8 — Erros

Identifique os 10 maiores erros absolutos.

Investigue os dados desses registros.

Existe alguma característica em comum?


In [ ]:
# Sua resposta



## Exercício 9 — Métricas

Explique, com suas palavras:

- MAE;
- MSE;
- RMSE;
- R².

Qual delas você considera mais fácil de interpretar para um gestor?
Justifique.


In [ ]:
# Sua resposta



## Exercício 10 — Decisão

Imagine dois modelos:

```text
Modelo A
MAE = 1.8 °C
R² = 0.82

Modelo B
MAE = 2.4 °C
R² = 0.90
```

Qual você escolheria?

Justifique considerando o contexto do problema.


In [ ]:
# Sua resposta



# 🚀 25. Desafio — Regressão no seu projeto

Agora procure identificar se o seu projeto possui um problema de **previsão numérica**.

Exemplos:

```text
Prever preço
Prever vendas
Prever consumo
Prever temperatura
Prever demanda
Prever tempo
Prever produtividade
Prever pontuação
```

### Etapa 1 — Defina o problema

```text
Quero prever:
____________________________
```

### Etapa 2 — Defina o target

```text
Target:
____________________________
```

### Etapa 3 — Escolha as features

Escolha variáveis que possam contribuir para a previsão.

### Etapa 4 — Treine

Utilize:

```text
LinearRegression
```

### Etapa 5 — Avalie

Apresente:

- MAE;
- MSE;
- RMSE;
- R².

### Etapa 6 — Visualize

Apresente pelo menos:

1. uma visualização de relação entre variáveis;
2. Real × Previsto;
3. distribuição dos erros.

### Etapa 7 — Conclusão

Responda:

> **O modelo é adequado para o problema?**

Justifique utilizando os resultados encontrados.


In [ ]:
# Desenvolva a regressão do seu projeto aqui.



# 🏭 26. Aplicação no projeto didático

Agora temos três possibilidades de modelagem:

```text
CLUSTERING
↓
Descobrir grupos

CLASSIFICAÇÃO
↓
Prever categorias

REGRESSÃO
↓
Prever valores numéricos
```

No exemplo industrial:

```text
Clustering
→ grupos de comportamento dos motores

Classificação
→ Normal / Falha

Regressão
→ Temperatura prevista
```

Esse conjunto de técnicas mostra que a escolha do algoritmo depende da **pergunta que queremos responder**.


# 📌 27. Checklist da Aula

- [ ] Sei diferenciar classificação e regressão;
- [ ] Sei identificar features e target;
- [ ] Sei aplicar Regressão Linear;
- [ ] Sei interpretar coeficientes;
- [ ] Sei fazer previsões;
- [ ] Sei calcular MAE;
- [ ] Sei calcular MSE;
- [ ] Sei calcular RMSE;
- [ ] Sei interpretar R²;
- [ ] Sei analisar erros;
- [ ] Sei analisar resíduos;
- [ ] Sei comparar modelos com diferentes conjuntos de variáveis;
- [ ] Entendo limitações da regressão linear;
- [ ] Consigo aplicar regressão ao meu projeto.

---

# 🎯 Conclusão

A sequência da disciplina está agora:

```text
Aula 03 → Pandas
Aula 04 → Limpeza
Aula 05 → ETL
Aula 06 → Web Scraping
Aula 07 → Banco de Dados + SQL
Aula 08 → Análise Exploratória
Aula 09 → Amostragem + Balanceamento
Aula 10 → Visualização
Aula 11 → Clustering
Aula 12 → Classificação
Aula 13 → Árvores de Decisão + Comparação
Aula 14 → Regressão
```

Já percorremos:

```text
DADOS
 ↓
PREPARAÇÃO
 ↓
EXPLORAÇÃO
 ↓
VISUALIZAÇÃO
 ↓
CLUSTERING
 ↓
CLASSIFICAÇÃO
 ↓
REGRESSÃO
```

Na próxima aula vamos sair um pouco da modelagem individual e trabalhar a **avaliação e validação dos modelos**, incluindo a ideia de validação cruzada e comparação mais confiável.

> 📊 **Próxima aula: Validação de Modelos e Seleção de Modelos.**
